# Phase 4A — chunk ANN on Colab T4

This does **not** train a new model. It embeds article chunks with the **same** frozen encoder used on the PC:
`paraphrase-multilingual-MiniLM-L12-v2`

**Before Run all:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload `clean_articles.csv` to Google Drive folder `ULTRA_Phase4A`

Progress is saved on Drive every 500 articles. If Colab disconnects, Run all again.

In [ ]:
# Cell 1 — GPU check + Drive
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save, then Runtime → Restart session.")

from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

In [ ]:
# Cell 2 — install libraries (1–2 minutes)
!pip -q install sentence-transformers chromadb pandas numpy

In [ ]:
# Cell 3 — paths (change only if your Drive folder name is different)
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/ULTRA_Phase4A")
CSV = DRIVE / "clean_articles.csv"
ART = DRIVE / "artifacts"
CHROMA_DIR = ART / "chroma_chunks"
ART.mkdir(parents=True, exist_ok=True)

print("Looking for:", CSV)
print("Exists:", CSV.is_file(), "size_MB:", round(CSV.stat().st_size / 1e6, 1) if CSV.is_file() else 0)
if not CSV.is_file():
    raise SystemExit(
        "CSV not found. On drive.google.com create folder ULTRA_Phase4A and upload clean_articles.csv into it."
    )

In [ ]:
# Cell 4 — build the chunk index (resumable). Keep this tab open.
import json, math, os, time
import torch
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
CHUNK_SIZE = 96
OVERLAP = 32
STRIDE = CHUNK_SIZE - OVERLAP
COLLECTION = "urdu_news_chunks_p4a"
EMBED_BATCH = 256
CHROMA_ADD = 2000

def n_chunks_for(n_tok):
    if n_tok <= CHUNK_SIZE:
        return 1
    return 1 + int(math.ceil((n_tok - CHUNK_SIZE) / float(STRIDE)))

def chunk_spans(n_tok):
    if n_tok <= 0:
        return [(0, 0)]
    if n_tok <= CHUNK_SIZE:
        return [(0, n_tok)]
    spans, start = [], 0
    while True:
        end = min(n_tok, start + CHUNK_SIZE)
        spans.append((start, end))
        if end >= n_tok:
            break
        start += STRIDE
    return spans

def chunk_texts(tok, text):
    ids = tok.encode(str(text or ""), add_special_tokens=False, truncation=False)
    spans = chunk_spans(len(ids))
    texts = [tok.decode(ids[a:b], skip_special_tokens=True) if b > a else "" for a, b in spans]
    return texts, spans

print("Loading corpus...")
df = pd.read_csv(CSV, encoding="utf-8-sig")
news = df["News Text"].fillna("").astype(str)
head = df["Headline"].fillna("").astype(str)
comb = df["combined_text"].fillna("").astype(str) if "combined_text" in df.columns else (head + " " + news)
n = len(comb)
print("articles:", n)

print("Loading MiniLM on GPU...")
model = SentenceTransformer(MODEL_NAME, device="cuda")
tok = model.tokenizer

len_path = ART / "corpus_token_lengths.npy"
if len_path.is_file():
    lengths = np.load(len_path)
    print("Loaded cached token lengths")
else:
    print("Tokenizing corpus (once)...")
    lengths = np.zeros(n, dtype=np.int32)
    t0 = time.perf_counter()
    for i, t in enumerate(comb):
        lengths[i] = len(tok.encode(str(t or ""), add_special_tokens=False, truncation=False))
        if (i + 1) % 20000 == 0:
            print("  tokenized", i + 1, "/%.1f min" % ((time.perf_counter() - t0) / 60))
    np.save(len_path, lengths)

nch = np.array([n_chunks_for(int(x)) for x in lengths], dtype=np.int32)
total = int(nch.sum())
print("expected chunks:", total)

emb_path = ART / "chunk_embeddings.f32"
aid_path = ART / "chunk_article_ids.npy"
start_path = ART / "chunk_starts.npy"
end_path = ART / "chunk_ends.npy"
prog_path = ART / "index_progress.json"

emb_mm = np.memmap(emb_path, dtype=np.float32, mode="w+" if not emb_path.is_file() else "r+", shape=(total, 384))
if aid_path.is_file() and np.load(aid_path).shape[0] == total:
    aids = np.load(aid_path)
    starts = np.load(start_path)
    ends = np.load(end_path)
else:
    aids = np.zeros(total, dtype=np.int32)
    starts = np.zeros(total, dtype=np.int32)
    ends = np.zeros(total, dtype=np.int32)

start_article, written = 0, 0
if prog_path.is_file():
    prog = json.loads(prog_path.read_text(encoding="utf-8"))
    start_article = int(prog.get("next_article", 0))
    written = int(prog.get("next_chunk", 0))
    print("Resuming at article", start_article, "chunk", written)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
col = client.get_or_create_collection(name=COLLECTION, metadata={"hnsw:space": "cosine"})
already = col.count()
print("Chroma count", already, "expected", total)

buf_ids, buf_emb, buf_meta = [], [], []
skip_chroma_until = already
t0 = time.perf_counter()
pending_txt, pending_loc = [], []

def flush():
    if not buf_ids:
        return
    col.add(ids=buf_ids, embeddings=buf_emb, metadatas=buf_meta)
    buf_ids.clear(); buf_emb.clear(); buf_meta.clear()

def encode_pending():
    global written
    if not pending_txt:
        return
    embs = np.asarray(
        model.encode(pending_txt, batch_size=EMBED_BATCH, convert_to_numpy=True, show_progress_bar=False),
        dtype=np.float32,
    )
    k = embs.shape[0]
    emb_mm[written: written + k] = embs
    for j, (art_i, cix, a, b) in enumerate(pending_loc):
        cid = written + j
        aids[cid] = art_i
        starts[cid] = a
        ends[cid] = b
        if cid >= skip_chroma_until:
            buf_ids.append("%s_%s" % (art_i, cix))
            buf_emb.append(embs[j].tolist())
            buf_meta.append({"article_id": int(art_i), "chunk_ix": int(cix), "start": int(a), "end": int(b)})
    written += k
    pending_txt.clear(); pending_loc.clear()
    if len(buf_ids) >= CHROMA_ADD:
        flush()

for i in range(start_article, n):
    texts, spans = chunk_texts(tok, comb.iloc[i])
    for j, ((a, b), tx) in enumerate(zip(spans, texts)):
        pending_txt.append(tx)
        pending_loc.append((i, j, a, b))
        if len(pending_txt) >= EMBED_BATCH:
            encode_pending()
    if (i + 1) % 500 == 0 or (i + 1) == n:
        encode_pending()
        flush()
        np.save(aid_path, aids); np.save(start_path, starts); np.save(end_path, ends)
        emb_mm.flush()
        prog_path.write_text(json.dumps({"next_article": i + 1, "next_chunk": written}), encoding="utf-8")
        mins = (time.perf_counter() - t0) / 60.0
        print("  articles %s/%s chunks=%s chroma=%s (%.1f min)" % (i + 1, n, written, col.count(), mins), flush=True)

encode_pending(); flush()
np.save(aid_path, aids); np.save(start_path, starts); np.save(end_path, ends)
emb_mm.flush()

def dir_size(p):
    p = Path(p)
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

info = {
    "n_articles": n,
    "n_chunks": int(written),
    "chroma_count": col.count(),
    "chunk_tokens": CHUNK_SIZE,
    "overlap": OVERLAP,
    "ann": "chromadb HNSW",
    "space": "cosine",
    "hnsw_params": "Chroma default HNSW, hnsw:space=cosine; M/ef not overridden",
    "embedding_dim": 384,
    "device": torch.cuda.get_device_name(0),
    "build_sec": round(time.perf_counter() - t0, 1),
    "chroma_bytes": dir_size(CHROMA_DIR),
    "embedding_memmap_bytes": emb_path.stat().st_size if emb_path.is_file() else 0,
}
(DRIVE / "index_build.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(json.dumps(info, indent=2))
if info["chroma_count"] != total:
    print("WARNING: chroma_count != expected chunks. Re-run this cell to resume.")
else:
    print("INDEX COMPLETE. Run the last cell to zip for download.")

In [ ]:
# Cell 5 — zip only the index (not the 515 MB CSV). Run after INDEX COMPLETE.
import json, shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/ULTRA_Phase4A")
info = json.loads((DRIVE / "index_build.json").read_text(encoding="utf-8"))
print("chroma_count:", info.get("chroma_count"), "n_chunks:", info.get("n_chunks"))
if info.get("chroma_count") != info.get("n_chunks"):
    raise SystemExit("Index not complete yet. Run Cell 4 again.")

print("Zipping artifacts to Drive (several minutes)...")
shutil.make_archive(str(DRIVE / "phase4a_chunk_index"), "zip", root_dir=str(DRIVE), base_dir="artifacts")
z = Path(str(DRIVE / "phase4a_chunk_index") + ".zip")
print("Created:", z, "MB:", round(z.stat().st_size / 1e6, 1))
print("From Google Drive folder ULTRA_Phase4A download BOTH:")
print("  1) phase4a_chunk_index.zip")
print("  2) index_build.json")